<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" alt="EDR|AI" width="300"/>

# Chapter 44 — Documents, Archives, and Text as Data

This is the **companion notebook** of [Chapter 44 — Documents, Archives, and Text as Data](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/text-as-data.html) from **EDR|AI — Evidence-Driven Research in the Age of AI**. Authored by [Davi Moreira](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html).

[Open the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/text-as-data.html) · [Book home](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html) · [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html)

*AI is your arm and your research assistant, not your brain.*

## How to use this notebook

1. Work top to bottom, with the chapter open in another tab.
2. Copy each **AI prompt** into your AI tool, run it, then record in the response cell what came back and what you verified.
3. Run the code cells; change something and run again.
4. Finish the **It is your turn** workspace at the end — that is this chapter's step of your own research project.
5. Log every AI use in your **AI Research Ledger**: task · tool · prompt · output summary · decision · verification method · remaining concern · you as the responsible researcher.
6. Your AI can be more than a chatbot: agentic tools can run multi-step work for you. Delegating boldly is fine; reviewing, curating, and deciding stay yours.

> **The research decision.** Decide which documents count as your corpus, what
> unit inside them you will code, and whose judgment turns words into a number:
> yours, a written codebook's, or a model's. A tool can read ten thousand pages
> overnight, but only you can say which pages belong and what a label means, so no
> tool gets to decide what your count measures.

## Code from the chapter

The cells below come from the chapter. Run them, then change something and run again — the numbers should move the way the chapter says they will.

*From the section “A worked example”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import pandas as pd

# Twelve constructed press-release sentences, hand-coded with the codebook:
# 1 = claims credit for money or a project delivered to the district.
releases = pd.DataFrame({"text": [
    "Rep. Alvarez secured $2.4 million for the Route 9 bridge repair.",
    "The senator announced a town hall on water quality next Tuesday.",
    "Thanks to my amendment, the county will receive new funding for rural clinics.",
    "I voted against the budget because it cuts funding for schools.",
    "Our office helped bring a new federal grant to the community college.",
    "The representative criticized the governor's plan to raise tolls.",
    "Fire stations across the district will get new equipment I fought for.",
    "The committee awarded the contract to the lowest bidder, a process I oppose.",
    "I am proud to deliver $800,000 for Main Street flood barriers.",
    "Statement on the passing of former mayor Lee.",
    "Senator Obi secured a meeting with the transit authority.",
    "New grant applications for small farms open Monday.",
], "human": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0]})

dictionary = ["secured", "funding", "grant", "awarded"]
releases["dict"] = releases.text.str.lower().apply(
    lambda t: int(any(word in t for word in dictionary)))

caught = ((releases.human == 1) & (releases.dict == 1)).sum()
missed = ((releases.human == 1) & (releases.dict == 0)).sum()
false_alarm = ((releases.human == 0) & (releases.dict == 1)).sum()
print(f"credit-claiming share, human codebook : {releases.human.mean()*100:.0f}%")
print(f"credit-claiming share, dictionary     : {releases.dict.mean()*100:.0f}%")
print(f"credit claims caught / missed         : {caught} / {missed}")
print(f"false alarms among the other releases : {false_alarm} of "
      f"{(releases.human == 0).sum()}")
print("\nthe dictionary counts words, not claims; its errors run in both")
print("directions, and only the hand-coded check shows how far")

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

*From the section “A seeded simulation”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 464
rng = np.random.default_rng(SEED)

# LEFT: two people code the same 200 releases. The true credit share is 15%.
n_pair = 200
truth_pair = rng.random(n_pair) < 0.15
coder_a = np.where(rng.random(n_pair) < 0.95, truth_pair, ~truth_pair)
coder_b = np.where(rng.random(n_pair) < 0.95, truth_pair, ~truth_pair)
coder_lazy = rng.random(n_pair) < 0.02          # marks almost nothing

def agreement(a, b):
    observed = np.mean(a == b)
    chance = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return observed, chance, (observed - chance) / (1 - chance)

# RIGHT: 6,000 releases, 20% claim credit. The AI labeler catches 75% of
# the credit claims and wrongly flags 3% of the rest.
N, n_gold = 6000, 300
truth = rng.random(N) < 0.20

fpc = 1 - n_gold / N                            # the subset is 5% of a fixed corpus

def run_once():
    ai = np.where(truth, rng.random(N) < 0.75, rng.random(N) < 0.03)
    gold = rng.permutation(N)[:n_gold]          # random subset a human codes
    p_ai = ai.mean()                            # AI share on every release
    se_ai = np.sqrt(p_ai * (1 - p_ai) / N)      # naive: labels taken as truth
    p_h = truth[gold].mean()
    se_h = np.sqrt(fpc * truth[gold].var(ddof=1) / n_gold)
    diff = truth[gold].astype(float) - ai[gold]  # human minus AI, per release
    p_c = p_ai + diff.mean()                    # AI share + average gap
    se_c = np.sqrt(fpc * diff.var(ddof=1) / n_gold)
    return (p_ai, se_ai), (p_h, se_h), (p_c, se_c)

first = run_once()
covers = np.array([[abs(p - truth.mean()) <= 1.96 * se for p, se in run_once()]
                   for _ in range(1000)])

for name, (a, b) in (("careful pair", (coder_a, coder_b)),
                     ("careful vs lazy", (coder_a, coder_lazy))):
    print(f"{name:16s} agreement {agreement(a, b)[0]:.2f}, chance "
          f"{agreement(a, b)[1]:.2f}, kappa {agreement(a, b)[2]:+.2f}")
print(f"\ntrue share in the corpus : {truth.mean()*100:.1f}%")
for name, (p, se), c in zip(("AI labels alone", "human subset alone",
                             "AI corrected by subset"), first, covers.mean(0)):
    print(f"{name:24s} : {p*100:.1f}% ({(p-1.96*se)*100:.1f} to "
          f"{(p+1.96*se)*100:.1f}); covers truth in {c*100:.0f}% of reruns")

fig, (axl, axr) = plt.subplots(1, 2, figsize=(10.4, 3.6))
for i, (a, b) in enumerate(((coder_a, coder_b), (coder_a, coder_lazy))):
    axl.bar([i - .26, i, i + .26], agreement(a, b), .26,
            color=["#2a78d6", "#9bbfe9", "#eb6834"])
axl.set_xticks([0, 1], ["Two careful coders", "Careful vs lazy"])
axl.axhline(0, color="#333333", lw=.6)
for y, (p, se), c in zip((2, 1, 0), first, ("#eb6834", "#777777", "#2a78d6")):
    axr.errorbar(p * 100, y, xerr=196 * se, fmt="o", color=c, capsize=3)
axr.axvline(truth.mean() * 100, color="#333333", ls="--", lw=1.2)
axr.set_yticks([2, 1, 0], ["AI alone", "Human subset", "AI corrected"])
axr.set_xlabel("Share of releases that claim credit (%)")
plt.show()

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

## It is your turn

<!-- station-pointer:begin -->
> **A further route beyond the five pathways.** This lesson
> extends [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html). Read it once
> you have declared your primary pathway and your question
> calls for this design. Studio 5's milestone asks for the
> same decisions, answered for this route.
<!-- station-pointer:end -->

*You have a declared design and a pathway. This route is where a question answered
from documents gets its corpus frame, its codebook, and its validation in writing,
before any count is reported.*

The hands-on half of this section lives in the chapter's **companion notebook**: open it in Colab with the badge at the top, and work the steps there.

Commit your own answer first, then delegate. Each prompt is a checkable job, not a
request for a verdict. Work them as a loop. The first answer is a draft: find the
claim you cannot check, say so in your next message, and run it again. Some tools
will label a whole corpus unattended and hand you a finished table. The finished
look is exactly what makes the questions in this chapter worth asking before you
accept it.

> **Do not delegate.**
>
> Three calls stay yours. You decide **which documents count as your corpus** and what
> the archive lost before you arrived. You decide **what each label means**, in a
> codebook you wrote, including the hard cases. And you decide **whether a machine's
> labels are good enough to count**, on the evidence of your own human-coded subset.
> A tool can read, sort, and flag at a speed no person matches. You own the definition,
> the check, and the sentence that reports the count.

**Step 1.** Write your corpus and its frame. In three lines, name the full body of documents
your question is about, the collection you can actually retrieve, and who wrote
those documents and why. Then list what may be missing and which way each gap
could tilt your count.

*When you are ready to delegate this step:*

```text
Act as an archivist. My question is about [the full set of documents]. The
collection I can retrieve is [your source, dates, how it was gathered]. List,
in a table, the kinds of documents likely missing from this collection, why
each went missing, and whether its absence would push my count up or down. Do
not tell me the collection is fine; I want the gaps.
```

After running, verify: check each named gap against what you know about how the
archive was built, and confirm that the gap you wrote down first is on the list.
Counters **illusion of completeness** (a tidy table that omits the one gap that
matters most).

✍️ **Your work for step 1.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 2.** Fix your unit and write the codebook entry for your main category. Give the
definition, at least two decision rules for hard cases, two examples, and two
non-examples, all taken from your own documents.

*When you are ready to delegate this step:*

```text
Here is my codebook entry: [paste it]. Act as a hostile second coder. Find
the five kinds of passages where two careful people applying this entry would
most likely disagree, and quote a made-up example of each, clearly labeled as
invented. Do not rewrite my definition; list the ambiguities so I fix them.
```

After running, verify: take each ambiguity to your own corpus and find a real
passage that shows it; if every objection is mild, push back with "assume my
definition is broken; name the worst case." Counters **sycophantic agreement**
(praise that reviews your ego, not your evidence).

✍️ **Your work for step 2.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 3.** Double-code a random subset and compute agreement yourself. Draw at least 50
units at random, and more if your main category is rare, because a kappa built
on a handful of positive cases swings widely. Have a second person code them from the codebook alone, and
report percent agreement, chance agreement, and Cohen's kappa for your main
category. Revise the codebook where you disagreed, and say so. If no second
person is available, recode the same units yourself a week later without
looking at your first labels, and report it as the weaker check it is.

*When you are ready to delegate this step:*

```text
Act as a Python tutor. Here are two columns of labels from two coders on the
same units: [paste]. Explain, step by step, how to compute percent agreement,
the agreement expected by chance, and Cohen's kappa, and write the code. Show
the intermediate numbers so I can check each one by hand.
```

After running, verify: compute the three numbers by hand from your own two
columns and confirm the code matches. A chance term that does not come from
each coder's own label rates, such as a flat 50 percent, is not Cohen's kappa. Counters **plausible-but-wrong method** (a
reliability number that looks standard and measures the wrong thing).

✍️ **Your work for step 3.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 4.** Validate any machine labeler against a human-coded gold subset before you count
with it. Code a fresh random subset yourself, run the labeler on the same units
with a frozen prompt, and report its misses and false alarms for each category.
If you use its labels for a share, correct the share with the subset and report
the interval.

*When you are ready to delegate this step:*

```text
Apply this codebook exactly: [paste it]. Label each of the following
documents, and for every label give the one sentence from the document that
justifies it, copied word for word. If no sentence justifies a label, write
"no quote" instead of paraphrasing. [paste documents with their IDs]
```

After running, verify: search each quoted sentence in its source document,
character for character, and count any quote you cannot find as a failed
label. Counters **confident fabrication** (a quote invented to fit a label
arrives as confidently as a real one).

✍️ **Your work for step 4.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 5.** Write the Research Contract lines for this route. State the question, the
estimand (a share, a trend, or a difference, and in which documents), the data
strategy (corpus frame, unit, sampling of the gold subset), the answer strategy
(codebook, labeler, correction), the warrant that licenses its reach, and the
limits: what the words cannot tell you about beliefs, deeds, or causes. If your question
is not a text question, write one line saying which part of your project, if
any, rests on documents, and whether its labels have been checked.

*When you are ready to delegate this step:*

```text
Here is the claim I plan to report from my text analysis: "[paste your
sentence]." Act as a hostile reviewer. Name every word in it that claims more
than a count of documents can support, such as beliefs, intentions, effects,
or a population my corpus frame does not reach. Do not rewrite it for me.
```

After running, verify: check each flagged word against your estimand and frame,
keep only the objections your design actually forces, and fix those yourself.
Counters **silent scope change** (a count of what was written quietly upgraded
to a claim about what people think or did).

✍️ **Your work for step 5.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 6.** Log the step in your AI Research Ledger, and verify at least one output with a
named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). This
route deserves two. **Alternative code** is the natural first check: recompute
the labeler's error table and the corrected share from your hand-coded subset
by a second route, such as a spreadsheet, and land on the same numbers. **Peer
reasoning** is the second: a second coder who reads only your codebook and
disagrees with you on a case is evidence about the codebook, not about the
coder. An AI reviewer may run the checks with you; the decision to accept or
reject stays yours.

✍️ **Your work for step 6.** Double-click this cell and write your answer here.

### The standard this section is held to

Use this as a self-check while you work. It is also the bar the same work meets later, once your project carries it. Each row: **0** missing, **1** attempted but incomplete, generic, or unverified, **2** complete, specific to your own project, and verified where a check applies. **14 points in all.**

| # | Criterion | 0–2 |
|---|---|---|
| Step 1 | Write your corpus and its frame | |
| Step 2 | Fix your unit and write the codebook entry for your main category | |
| Step 3 | Double-code a random subset and compute agreement yourself | |
| Step 4 | Validate any machine labeler against a human-coded gold subset before you count with it | |
| Step 5 | Write the Research Contract lines for this route | |
| Step 6 | Log the step in your AI Research Ledger, and verify at least one output with a named method from the Verification Guide | |
| + | Craft and verification record: AI use logged in your AI Research Ledger, claims stated with their uncertainty, and each key claim verified with a named method | |

In [ ]:
# Scratch space — use this cell for any code your steps need.

**Before you leave this notebook:** add today's rows to your AI Research Ledger, and verify your key claim with a named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). AI can review AI — but the last decision is human.

Next: [Chapter 45 — Qualitative Inquiry: Interviews, Documents, and Themes](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/qualitative-inquiry.html). That chapter may not be on your route — [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html) is the junction; follow the lesson that matches your own project.